In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch04-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2",
            "mordredcommunity==2.0.7", "networkx==3.6.1"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "04-04-mordred.py"

# 哪些 Mordred 欄位能區分這批分子？
從 ESOL 的骨架切分中抽出 48 筆訓練結構與 12 筆驗證結構。首次執行時，DeepChem 會下載 ESOL。圖中的數字記錄每個步驟後剩餘的欄數。要做練習，可以把程式加在下一格的末尾，再重跑該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
from time import perf_counter
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mordred import Calculator, descriptors
from rdkit import Chem

# ESOL：1,128 筆，水溶解度 log10(mol/L)，原始研究 DOI 10.1021/ci034243x。
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
_, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
rng = np.random.default_rng(42)
calc = Calculator(descriptors, ignore_3D=True)

def sample_table(split, n):
    indices = rng.choice(len(split), size=n, replace=False)
    smiles = split.ids[indices]
    mols = [Chem.MolFromSmiles(smi) for smi in smiles]
    assert all(mol is not None for mol in mols)
    raw = calc.pandas(mols, nproc=1, quiet=True)
    raw.index = smiles
    table = raw.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return raw, table

start = perf_counter()
raw_train, X = sample_table(train, 48)
print(f"{len(X)} training structures x {X.shape[1]} descriptors: {perf_counter() - start:.1f} s")
complete = X.dropna(axis="columns")
varying = complete.loc[:, complete.nunique() > 1]
unique = varying.T.drop_duplicates().T
print("Registered, complete, varying, distinct:", X.shape[1], complete.shape[1],
      varying.shape[1], unique.shape[1])
kept = unique.columns
_, X_valid = sample_table(valid, 12)
validation = X_valid.loc[:, kept]
print("Missing validation cells before filling:", int(validation.isna().sum().sum()))
validation = validation.fillna(unique.median())
assert validation.columns.equals(kept) and not validation.isna().any().any()
print("Validation table:", validation.shape)
print("Training audit:")
for col in ["nO", "AATS8m", "nI", "nBr", "nCl"]:
    print(col, "missing", X[col].isna().sum(), "distinct finite values", X[col].nunique())
missing_examples = raw_train.loc[X["AATS8m"].isna(), "AATS8m"]
if len(missing_examples):
    print("AATS8m example:", missing_examples.index[0], missing_examples.iloc[0])
counts = [X.shape[1], complete.shape[1], varying.shape[1], unique.shape[1]]
labels = ["Registered (2D)", "Complete on train", "Nonconstant", "Distinct columns"]
fig, ax = plt.subplots(figsize=(7.4, 3.3), layout="constrained")
bars = ax.barh(labels, counts, color=["#52657a", "#39798c", "#257667", "#155549"])
ax.invert_yaxis()
ax.set(xlabel="Number of descriptor columns", xlim=(0, max(counts) * 1.13),
       title=f"ESOL descriptor audit | {len(X)} training molecules")
ax.bar_label(bars, padding=5)
ax.spines[["top", "right", "left"]].set_visible(False)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "04-04-mordred.png", dpi=160)
plt.close(fig)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "04-04-mordred.png")))